In [8]:
# vllm serve Qwen/Qwen3-0.6B --dtype float16 --gpu-memory-utilization 0.80

import warnings
warnings.filterwarnings("ignore")

import time, requests, json, os, math, sys

VLLM_URL = "http://localhost:8000"
os.makedirs("outputs", exist_ok=True)

print("Waiting for vLLM server...")
for attempt in range(60):
    try:
        r = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        if r.status_code == 200:
            MODEL = r.json()["data"][0]["id"]
            break
    except requests.ConnectionError:
        pass
    time.sleep(5)
    if attempt % 6 == 5:
        print(f"  Still waiting... ({(attempt + 1) * 5}s elapsed)")
else:
    raise RuntimeError(
        "vLLM server not reachable after 5 minutes."
    )

print(f"Connected to {VLLM_URL} — model: {MODEL}")

Waiting for vLLM server...
Connected to http://localhost:8000 — model: Qwen/Qwen3-0.6B


In [9]:
from openai import OpenAI
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")

In [10]:
start = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", 
               "content": "What is PagedAttention in one sentence?"}],
    max_tokens=80,
    temperature=0.7,
    top_p=0.8,
    extra_body={"top_k": 20, 
                "chat_template_kwargs": {"enable_thinking": False}},
)
elapsed = time.time() - start

print(f"Response ({elapsed:.2f}s, {resp.usage.completion_tokens} tokens):")
print(resp.choices[0].message.content)
print(f"\nUsage: {resp.usage.prompt_tokens} prompt + "
      f"{resp.usage.completion_tokens} completion = {resp.usage.total_tokens} total")

Response (0.38s, 37 tokens):
Paged Attention is a technique in neural networks that improves the efficiency of attention mechanisms by breaking the attention process into smaller, manageable blocks, allowing for parallel processing and reduced computational overhead.

Usage: 21 prompt + 37 completion = 58 total


In [11]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "The capital of France is"}],
    max_tokens=15,
    temperature=0.0,
    logprobs=True,
    top_logprobs=5,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(f"Response: {resp.choices[0].message.content.strip()}\n")
print("Token-by-token probabilities:\n")

for tok in resp.choices[0].logprobs.content[:8]:
    print(f"  Chosen: '{tok.token}'  (logprob {tok.logprob:.2f})")
    if tok.top_logprobs:
        for alt in tok.top_logprobs[:5]:
            pct = math.exp(alt.logprob) * 100
            bar = "\u2588" * min(20, max(1, int(pct / 5)))
            print(f"    {pct:5.1f}%  {bar}  '{alt.token}'")
    print()

Response: The capital of France is **Paris**.

Token-by-token probabilities:

  Chosen: 'The'  (logprob -0.00)
     99.9%  ███████████████████  'The'
      0.1%  █  'France'
      0.0%  █  'Capital'
      0.0%  █  'Paris'
      0.0%  █  'Le'

  Chosen: ' capital'  (logprob -0.00)
    100.0%  ███████████████████  ' capital'
      0.0%  █  ' Capital'
      0.0%  █  ' **'
      0.0%  █  ' French'
      0.0%  █  '**'

  Chosen: ' of'  (logprob -0.00)
    100.0%  ███████████████████  ' of'
      0.0%  █  ' city'
      0.0%  █  ' is'
      0.0%  █  ' ('
      0.0%  █  ' and'

  Chosen: ' France'  (logprob -0.00)
    100.0%  ███████████████████  ' France'
      0.0%  █  ' **'
      0.0%  █  ' the'
      0.0%  █  'France'
      0.0%  █  ' Europe'

  Chosen: ' is'  (logprob -0.00)
    100.0%  ███████████████████  ' is'
      0.0%  █  ' in'
      0.0%  █  ','
      0.0%  █  ' was'
      0.0%  █  ' **'

  Chosen: ' **'  (logprob -0.09)
     91.0%  ██████████████████  ' **'
      7.7%  █  ' Paris'

In [12]:
def get_vllm_metrics(base_url=VLLM_URL):
    """Scrape vLLM Prometheus /metrics and return {name: value}."""
    r = requests.get(f"{base_url}/metrics")
    metrics = {}
    for line in r.text.split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        name = line.split("{")[0].split()[0]
        try:
            metrics[name] = float(line.split()[-1])
        except (ValueError, IndexError):
            continue
    return metrics

metrics = get_vllm_metrics()
print("Current vLLM Metrics:")
for key in ["vllm:num_requests_running", "vllm:num_requests_waiting",
            "vllm:gpu_cache_usage_perc", "vllm:cpu_cache_usage_perc",
            "vllm:prompt_tokens_total", "vllm:generation_tokens_total"]:
    if key in metrics:
        print(f"  {key.replace('vllm:', '')}: {metrics[key]:g}")

with open("outputs/metrics_snapshot.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nFull metrics saved to outputs/metrics_snapshot.json")

Current vLLM Metrics:
  num_requests_running: 0
  num_requests_waiting: 0
  prompt_tokens_total: 76
  generation_tokens_total: 84

Full metrics saved to outputs/metrics_snapshot.json


In [13]:
import concurrent.futures

prompts = [
    "What is quantization?",
    "Explain KV caching briefly.",
    "What is continuous batching?",
    "Why is LLM inference memory-bound?",
    "What is PagedAttention?",
]

def _ask(prompt):
    return client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=60, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )

before = get_vllm_metrics()
print(f"Sending {len(prompts)} concurrent requests...\n")
start = time.time()

with concurrent.futures.ThreadPoolExecutor(
    max_workers=len(prompts)) as pool:
    futures = {pool.submit(_ask, p): p for p in prompts}
    time.sleep(0.5)
    during = get_vllm_metrics()
    running = during.get("vllm:num_requests_running", "--")
    waiting = during.get("vllm:num_requests_waiting", "--")
    print(f"  [mid-flight]  running: {running}  |  waiting: {waiting}")

    for f in concurrent.futures.as_completed(futures):
        resp = f.result()
        print(f"  done: \"{futures[f][:40]}\" -> {resp.usage.completion_tokens} tokens")

elapsed = time.time() - start
after = get_vllm_metrics()
tokens = after.get("vllm:generation_tokens_total", 0) - before.get(
    "vllm:generation_tokens_total", 0)

print(f"\nAll {len(prompts)} completed in {elapsed:.2f}s")
if tokens > 0:
    print(f"Tokens generated: {tokens:g}  |  ~{tokens / elapsed:.1f} tokens/s")

Sending 5 concurrent requests...

  [mid-flight]  running: 5.0  |  waiting: 0.0
  done: "Why is LLM inference memory-bound?" -> 60 tokens
  done: "What is PagedAttention?" -> 60 tokens
  done: "Explain KV caching briefly." -> 60 tokens
  done: "What is continuous batching?" -> 60 tokens
  done: "What is quantization?" -> 60 tokens

All 5 completed in 0.69s
Tokens generated: 300  |  ~436.1 tokens/s


In [14]:
for f in concurrent.futures.as_completed(futures):
        resp = f.result()
        prompt_text = futures[f]
        answer_text = resp.choices[0].message.content
        
        print(f"\nPrompt: {prompt_text}")
        print(f"Answer: {answer_text}")
        print(f"(Tokens: {resp.usage.completion_tokens})")


Prompt: What is PagedAttention?
Answer: Paged Attention is a technique used in transformer architectures, particularly in the context of **Transformer-based models** (such as BERT, RoBERTa, etc.), to improve efficiency and performance. It is a method that breaks the attention calculation into smaller, manageable chunks, which reduces the computational complexity and
(Tokens: 60)

Prompt: Explain KV caching briefly.
Answer: KV caching is a technique used in distributed systems to store frequently accessed keys in a **cache**, typically a **key-value (KV) database**. The goal is to minimize the number of times a system has to access a key, thereby improving performance and reducing the load on the database.

### Key
(Tokens: 60)

Prompt: Why is LLM inference memory-bound?
Answer: The **inference memory-boundness** of large language models (LLMs) is a complex and multifaceted issue, primarily due to the following reasons:

---

### 1. **Memory Usage and Storage**
- **Large Vocabulary and

In [16]:
SYSTEM_PROMPT = (
    "You are a helpful AI teaching assistant for a course on "
    "LLM optimization. You specialize in explaining concepts like "
    "quantization, inference optimization, and model serving. Keep "
    "answers concise -- one or two sentences."
)

questions = [
    "What is weight quantization?",
    "How does vLLM handle memory?",
    "What is continuous batching?",
    "Why use prefix caching?",
    "What is GPTQ?",
]

before = get_vllm_metrics()
timings = []
tok_counts = []

print("Sending 5 requests with the SAME system prompt...\n")
for i, q in enumerate(questions):
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ],
        max_tokens=60, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    dt = time.time() - t0
    timings.append(dt)
    tok_counts.append(resp.usage.completion_tokens)
    tokens = resp.usage.completion_tokens
    ms_per_tok = (dt / tokens * 1000) if tokens > 0 else 0.0
    print(f"  [{i+1}] {q:<35} {dt:.2f}s  ({tokens} tok, {ms_per_tok:.0f} ms/tok)")
    
after = get_vllm_metrics()

prefix_before = before.get("vllm:prefix_cache_queries_total", 0)
prefix_after = after.get("vllm:prefix_cache_queries_total", 0)

print(f"\nPrefix cache queries: {prefix_before:g} -> {prefix_after:g}  (+{prefix_after - prefix_before:g})")

cache_keys = [k for k in after if "prefix" in k.lower() 
              or "cache_hit" in k.lower()]
for k in sorted(cache_keys):
    b, a = before.get(k, 0), after.get(k, 0)
    if a != b and k != "vllm:prefix_cache_queries_total":
        print(f"  {k}: {b:g} -> {a:g}")

print("\n The increasing prefix_cache_queries count confirms vLLM is ")
print("checking and reusing cached KV blocks for the shared system prompt.")

Sending 5 requests with the SAME system prompt...

  [1] What is weight quantization?        4.45s  (48 tok, 93 ms/tok)
  [2] How does vLLM handle memory?        2.33s  (25 tok, 93 ms/tok)
  [3] What is continuous batching?        3.32s  (36 tok, 92 ms/tok)
  [4] Why use prefix caching?             3.25s  (35 tok, 93 ms/tok)
  [5] What is GPTQ?                       4.92s  (53 tok, 93 ms/tok)

Prefix cache queries: 8368 -> 8683  (+315)
  vllm:prefix_cache_hits_total: 1904 -> 2160

 The increasing prefix_cache_queries count confirms vLLM is 
checking and reusing cached KV blocks for the shared system prompt.
